<a href="https://colab.research.google.com/github/bihagkashikar/bits-pilani-mtech-genai-ml/blob/master/maths-assignment-01/q1_linear_systems.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Q1: Finding Solutions of Linear Systems
This notebook contains the complete self-contained Python solution for Q1.

## Q1.1: Augmented Matrix, REF and RREF


In [21]:
"""Q1.1: Construct [A | b] and compute REF and RREF without linear-algebra built-ins."""

import random


def copy_matrix(matrix):
    """Return a manual copy of a matrix."""
    copied = []
    for row in matrix:
        copied.append([value for value in row])
    return copied


def print_matrix(title, matrix):
    """Print every matrix entry with eight decimal places."""
    print(title)
    for row in matrix:
        print("[" + ", ".join(f"{value:.8f}" for value in row) + "]")


def augmented_matrix(matrix_a, vector_b):
    """Construct the augmented matrix [A | b]."""
    augmented = []
    for row_index in range(len(matrix_a)):
        augmented.append(matrix_a[row_index][:] + [vector_b[row_index]])
    return augmented


def row_echelon_form(matrix):
    """Reduce a matrix to REF using row swaps and elementary row operations."""
    result = copy_matrix(matrix)
    row_count = len(result)
    column_count = len(result[0])
    pivot_row = 0
    pivot_columns = []
    for column in range(column_count):
        if pivot_row == row_count:
            break
        selected_row = pivot_row
        while selected_row < row_count and abs(result[selected_row][column]) < 1e-12:
            selected_row += 1
        if selected_row == row_count:
            continue
        result[pivot_row], result[selected_row] = result[selected_row], result[pivot_row]
        pivot = result[pivot_row][column]
        for lower_row in range(pivot_row + 1, row_count):
            multiplier = result[lower_row][column] / pivot
            for entry in range(column, column_count):
                result[lower_row][entry] -= multiplier * result[pivot_row][entry]
        pivot_columns.append(column)
        pivot_row += 1
    return result, pivot_columns


def reduced_row_echelon_form(matrix):
    """Reduce a matrix to RREF and explicitly handle a possible zero pivot."""
    result, pivot_columns = row_echelon_form(matrix)
    for pivot_row in range(len(pivot_columns) - 1, -1, -1):
        pivot_column = pivot_columns[pivot_row]
        pivot = result[pivot_row][pivot_column]
        # If division by zero is encountered for user input, choose a different A and/or b.
        if abs(pivot) < 1e-12:
            continue
        for entry in range(len(result[0])):
            result[pivot_row][entry] /= pivot
        for upper_row in range(pivot_row):
            multiplier = result[upper_row][pivot_column]
            for entry in range(len(result[0])):
                result[upper_row][entry] -= multiplier * result[pivot_row][entry]
    return result, pivot_columns

In [18]:
"""Q1.1 input section: read A and b from the user and compute REF and RREF."""


def read_decimal_values(prompt, expected_count):
    """Read exactly expected_count decimal values from one input line."""
    values = input(prompt).split()
    if len(values) != expected_count:
        raise ValueError(f"Expected {expected_count} values, but received {len(values)}.")
    return [float(value) for value in values]


def read_system_input():
    """Read m, n, matrix A, and vector b with m < n."""
    row_count = int(input("Enter the number of rows m: "))
    column_count = int(input("Enter the number of columns n: "))
    if row_count <= 0 or column_count <= row_count:
        raise ValueError("The dimensions must satisfy m > 0 and m < n.")

    matrix_a = []
    for row_index in range(row_count):
        prompt = f"Enter {column_count} decimal values for row {row_index + 1} of A: "
        matrix_a.append(read_decimal_values(prompt, column_count))

    vector_b = read_decimal_values(
        f"Enter {row_count} decimal values for vector b: ", row_count
    )
    return row_count, column_count, matrix_a, vector_b


# Enter values such as 3.47681539, separated by spaces, when prompted.
row_count, column_count, matrix_a, vector_b = read_system_input()
augmented = augmented_matrix(matrix_a, vector_b)
ref, _ = row_echelon_form(augmented)
rref, _ = reduced_row_echelon_form(augmented)
print("\n===== USER INPUT =====")
print(f"Dimensions: m = {row_count}, n = {column_count}")
print_matrix("Matrix A entered by the user", matrix_a)
print_matrix("Vector b entered by the user", [[value] for value in vector_b])
print("\n===== CALCULATED RESULTS =====")
print_matrix("Augmented matrix [A | b]", augmented)
print_matrix("REF", ref)
print_matrix("RREF", rref)


===== USER INPUT =====
Dimensions: m = 2, n = 3
Matrix A entered by the user
[2.50000000, -1.20000000, 3.70000000]
[1.10000000, 4.80000000, -2.60000000]
Vector b entered by the user
[5.30000000]
[-1.70000000]

===== CALCULATED RESULTS =====
Augmented matrix [A | b]
[2.50000000, -1.20000000, 3.70000000, 5.30000000]
[1.10000000, 4.80000000, -2.60000000, -1.70000000]
REF
[2.50000000, -1.20000000, 3.70000000, 5.30000000]
[0.00000000, 5.32800000, -4.22800000, -4.03200000]
RREF
[1.00000000, 0.00000000, 1.09909910, 1.75675676]
[0.00000000, 1.00000000, -0.79354354, -0.75675676]


## Q1.2: Pivot and Non-Pivot Columns


In [22]:
"""Q1.2: Identify pivot/non-pivot columns and construct solution vectors."""


def identify_columns(rref, pivot_columns, variable_count):
    """Return pivot and non-pivot variable columns from RREF([A | b])."""
    pivot_variables = [column for column in pivot_columns if column < variable_count]
    non_pivot_variables = []
    for column in range(variable_count):
        if column not in pivot_variables:
            non_pivot_variables.append(column)
    return pivot_variables, non_pivot_variables


def particular_solution(rref, pivot_columns, variable_count):
    """Find one solution by setting every free variable to zero."""
    solution = [0.0 for _ in range(variable_count)]
    for row, pivot_column in enumerate(pivot_columns):
        if pivot_column < variable_count:
            solution[pivot_column] = rref[row][variable_count]
    return solution


def null_space_vectors(rref, pivot_columns, variable_count):
    """Build one homogeneous solution vector for each non-pivot column."""
    pivot_variables = [column for column in pivot_columns if column < variable_count]
    free_variables = []
    for column in range(variable_count):
        if column not in pivot_variables:
            free_variables.append(column)
    vectors = []
    for free_column in free_variables:
        vector = [0.0 for _ in range(variable_count)]
        vector[free_column] = 1.0
        for row, pivot_column in enumerate(pivot_variables):
            vector[pivot_column] = -rref[row][free_column]
        vectors.append(vector)
    return vectors

## Q1.3: Random 5 x 7 Example and Verification


In [23]:
"""Q1.3: Generate a random 5 x 7 example and verify the general solution."""


def multiply_matrix_vector(matrix, vector):
    """Multiply a matrix and vector using explicit loops."""
    result = []
    for row in matrix:
        total = 0.0
        for column in range(len(vector)):
            total += row[column] * vector[column]
        result.append(total)
    return result


def add_vectors(first, second):
    """Add two vectors entry by entry."""
    return [first[index] + second[index] for index in range(len(first))]


def scale_vector(scalar, vector):
    """Multiply a vector by a scalar entry by entry."""
    return [scalar * value for value in vector]


def maximum_absolute_difference(first, second):
    """Return the largest absolute difference between two vectors."""
    largest = 0.0
    for index in range(len(first)):
        difference = abs(first[index] - second[index])
        if difference > largest:
            largest = difference
    return largest


def print_vector(title, vector):
    """Print a vector with a clear label and eight decimal places."""
    print(title)
    print("[" + ", ".join(f"{value:.8f}" for value in vector) + "]")


# Generate a random 5 x 7 system with decimal entries.
random.seed(41601)
row_count = 5
column_count = 7
matrix_a = []
for _ in range(row_count):
    matrix_a.append([random.uniform(-9.0, 9.0) for _ in range(column_count)])
known_solution = [random.uniform(-9.0, 9.0) for _ in range(column_count)]
vector_b = multiply_matrix_vector(matrix_a, known_solution)
augmented = augmented_matrix(matrix_a, vector_b)
ref, _ = row_echelon_form(augmented)
rref, pivot_columns = reduced_row_echelon_form(augmented)
pivot, non_pivot = identify_columns(rref, pivot_columns, column_count)
particular = particular_solution(rref, pivot_columns, column_count)
homogeneous = null_space_vectors(rref, pivot_columns, column_count)

print("\n========== INPUT SYSTEM ==========")
print(f"Dimensions: A is {row_count} x {column_count}")
print_matrix("Matrix A", matrix_a)
print_vector("Vector b", vector_b)
print_vector("Known solution used to construct b", known_solution)

print("\n========== ROW REDUCTION ==========")
print_matrix("Augmented matrix [A | b]", augmented)
print_matrix("REF", ref)
print_matrix("RREF", rref)

print("\n========== PIVOT INFORMATION ==========")
print(f"Pivot columns (1-based): {[column + 1 for column in pivot]}")
print(f"Non-pivot columns (1-based): {[column + 1 for column in non_pivot]}")

print("\n========== SOLUTION VECTORS ==========")
print_vector("Particular solution x_p", particular)
for index, vector in enumerate(homogeneous, start=1):
    print_vector(f"Null-space vector v_{index} for Ax = 0", vector)

coefficients = [random.uniform(-9.0, 9.0) for _ in homogeneous]
general_solution = particular[:]
for index in range(len(homogeneous)):
    general_solution = add_vectors(
        general_solution, scale_vector(coefficients[index], homogeneous[index])
    )
reconstructed_b = multiply_matrix_vector(matrix_a, general_solution)

print("\n========== GENERAL SOLUTION ==========")
print_vector("Free-variable coefficients", coefficients)
print_vector("General solution x", general_solution)

print("\n========== VERIFICATION ==========")
print_vector("A times general solution", reconstructed_b)
print(
    "Maximum absolute error |A x - b|:",
    f"{maximum_absolute_difference(reconstructed_b, vector_b):.8e}",
)


========== INPUT SYSTEM ==========
Dimensions: A is 5 x 7
Matrix A
[6.76703566, -2.24561272, -7.71899173, -0.98864788, -8.22867733, 0.90988703, 8.89195161]
[7.35468136, 1.64258069, -8.90210539, 5.44640379, -4.70918894, -5.65638362, 8.94445397]
[-0.62190896, -7.48498027, 4.30460653, -2.57414307, -3.93788934, 6.20677215, 8.49658497]
[6.61771988, -6.27900627, -0.18379768, 3.53495029, 8.59276677, -2.33772482, 1.67364233]
[-4.05019750, -3.25792681, -6.00088794, 2.65704361, 5.43975658, 8.29997879, -1.08913688]
Vector b
[-30.83347132, 4.80159952, 27.02019873, 142.58152152, -43.24458984]
Known solution used to construct b
[5.36632842, -3.49486864, 5.60339775, -0.96993640, 8.33336247, -4.47382401, 4.49183192]

========== ROW REDUCTION ==========
Augmented matrix [A | b]
[6.76703566, -2.24561272, -7.71899173, -0.98864788, -8.22867733, 0.90988703, 8.89195161, -30.83347132]
[7.35468136, 1.64258069, -8.90210539, 5.44640379, -4.70918894, -5.65638362, 8.94445397, 4.80159952]
[-0.62190896, -7.4849802